# 13. 어린이 교통사고 고위험지역 예측 AI

## 목표
과거 사고 패턴, 어린이 인구, 자동차 등록대수를 이용해 **다음 연도에 상대적으로 어린이 교통사고 위험이 높은 시군구**를 예측한다.

### 고위험의 정의
각 목표 연도에서 `어린이 1만 명당 사고율`이 **상위 25%**인 지역을 고위험(1), 나머지를 0으로 정의한다.

### 검증 방식
랜덤 K-Fold가 아니라 시간 순서를 지키는 **Walk-forward validation**을 사용한다.
- 2022 학습 → 2023 평가
- 2022~2023 학습 → 2024 평가
- 2022~2024 학습 → 2025 평가

미래 정보가 과거 학습에 들어가는 데이터 누출을 줄이기 위한 방식이다.


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, precision_score, recall_score, accuracy_score
from sklearn.inspection import permutation_importance
import joblib

BASE = Path("..")
panel = pd.read_csv(BASE / "data_processed" / "panel_2020_2025_229.csv")


### 1. 과거 정보로 다음 연도 특징 만들기
목표 연도 자체의 사고정보는 입력에 사용하지 않는다.


In [2]:
panel = panel.sort_values(["sido_sgg_nm", "year"]).copy()
g = panel.groupby("sido_sgg_nm")

panel["prev_child_population"] = g["child_population_0_12"].shift(1)
panel["prev_vehicle_count"] = g["vehicle_count"].shift(1)
panel["prev_accident_count"] = g["accident_count"].shift(1)
panel["prev_injury_count"] = g["injury_count"].shift(1)
panel["prev_death_count"] = g["death_count"].shift(1)
panel["prev_accident_per_10k_child"] = g["accident_per_10k_child"].shift(1)
panel["prev_vehicles_per_100_child"] = g["vehicles_per_100_child"].shift(1)
panel["prev2_accident_count"] = g["accident_count"].shift(2)
panel["prev2_accident_rate"] = g["accident_per_10k_child"].shift(2)

panel["accident_count_change_1y"] = panel["prev_accident_count"] - panel["prev2_accident_count"]
panel["accident_rate_change_1y"] = panel["prev_accident_per_10k_child"] - panel["prev2_accident_rate"]
panel["accident_count_mean_2y"] = (panel["prev_accident_count"] + panel["prev2_accident_count"]) / 2

model_df = panel[panel["year"].between(2022, 2025)].dropna().copy()
model_df["high_risk_threshold_q75"] = model_df.groupby("year")["accident_per_10k_child"].transform(lambda x: x.quantile(0.75))
model_df["high_risk_label"] = (model_df["accident_per_10k_child"] >= model_df["high_risk_threshold_q75"]).astype(int)
model_df = model_df.rename(columns={"year":"target_year", "accident_count":"target_accident_count", "accident_per_10k_child":"target_accident_per_10k_child"})

feature_cols = [
    "prev_child_population", "prev_vehicle_count", "prev_accident_count",
    "prev_injury_count", "prev_death_count", "prev_accident_per_10k_child",
    "prev_vehicles_per_100_child", "accident_count_change_1y",
    "accident_rate_change_1y", "accident_count_mean_2y"
]
keep = ["target_year", "sido_sgg_nm"] + feature_cols + ["target_accident_count", "target_accident_per_10k_child", "high_risk_threshold_q75", "high_risk_label"]
model_df[keep].to_csv(BASE / "data_processed" / "ai_model_dataset_2022_2025.csv", index=False, encoding="utf-8-sig")
print(model_df.shape)
print(model_df.groupby("target_year")["high_risk_label"].mean())


(916, 23)
target_year
2022    0.253275
2023    0.253275
2024    0.253275
2025    0.253275
Name: high_risk_label, dtype: float64


### 2. 세 모델 비교
- Logistic Regression: 단순하고 해석이 쉬운 기준모델
- Random Forest: 비선형 관계를 학습
- HistGradientBoosting: 표 형태 데이터에서 강한 부스팅 모델


In [3]:
models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=3000, class_weight="balanced", random_state=42))
    ]),
    "Random Forest": RandomForestClassifier(
        n_estimators=500, max_depth=5, min_samples_leaf=4,
        class_weight="balanced", random_state=42
    ),
    "HistGradientBoosting": HistGradientBoostingClassifier(
        max_iter=250, max_leaf_nodes=15, learning_rate=0.05,
        l2_regularization=1.0, random_state=42
    )
}

results = []
oof = []
for test_year in [2023, 2024, 2025]:
    train = model_df[model_df["target_year"] < test_year]
    test = model_df[model_df["target_year"] == test_year]
    X_train, y_train = train[feature_cols], train["high_risk_label"]
    X_test, y_test = test[feature_cols], test["high_risk_label"]

    for name, model in models.items():
        model.fit(X_train, y_train)
        prob = model.predict_proba(X_test)[:, 1]
        pred = (prob >= 0.5).astype(int)
        results.append({
            "test_year": test_year, "model": name,
            "n_train": len(train), "n_test": len(test),
            "roc_auc": roc_auc_score(y_test, prob),
            "average_precision": average_precision_score(y_test, prob),
            "f1": f1_score(y_test, pred),
            "precision": precision_score(y_test, pred),
            "recall": recall_score(y_test, pred),
            "accuracy": accuracy_score(y_test, pred)
        })
        temp = test[["target_year", "sido_sgg_nm", "high_risk_label", "target_accident_per_10k_child"]].copy()
        temp["model"] = name
        temp["predicted_probability"] = prob
        temp["predicted_label"] = pred
        oof.append(temp)

metrics = pd.DataFrame(results)
avg = metrics.groupby("model")[["roc_auc", "average_precision", "f1", "precision", "recall", "accuracy"]].mean().reset_index()
display(avg.sort_values("average_precision", ascending=False))
metrics.to_csv(BASE / "results" / "model_walk_forward_metrics.csv", index=False, encoding="utf-8-sig")
avg.to_csv(BASE / "results" / "model_average_metrics.csv", index=False, encoding="utf-8-sig")
pd.concat(oof).to_csv(BASE / "results" / "oof_predictions_2023_2025.csv", index=False, encoding="utf-8-sig")


C:\Users\yudan\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] 지정된 파일을 찾을 수 없습니다
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "C:\Users\yudan\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
        "wmic CPU Get NumberOfCores /Format:csv".split(),
        capture_output=True,
        text=True,
    )
  File "C:\Users\yudan\anaconda3\Lib\subprocess.py", line 554, in run
    with Popen(*popenargs, **kwargs) as process:
         ~~~~~^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\yudan\anaconda3\Lib\subprocess.py", line 1039, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
    ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^


,model,roc_auc,average_precision,f1,precision,recall,accuracy
1,Logistic Regression,0.875109,0.666342,0.644020,0.533028,0.816092,0.771470
2,Random Forest,0.843282,0.586326,0.630558,0.521740,0.798851,0.761281
0,HistGradientBoosting,0.822461,0.519357,0.563149,0.544077,0.591954,0.768559


### 3. 최종 모델 선택 및 2026 상대적 고위험도 예측
Average Precision(PR-AUC)이 가장 높은 모델을 자동 선택한다. 2026년 실제 사고결과는 아직 없으므로, 아래 값은 **다음 연도 고위험지역 우선순위 점수**로 사용한다.


In [4]:
best_name = avg.sort_values(["average_precision", "roc_auc"], ascending=False).iloc[0]["model"]
best_model = models[best_name]
best_model.fit(model_df[feature_cols], model_df["high_risk_label"])
print("최종 모델:", best_name)
joblib.dump({"model": best_model, "feature_cols": feature_cols, "model_name": best_name}, BASE / "models" / "high_risk_classifier.joblib")

# 2025와 2024 자료로 2026 입력 특징 구성
p25 = panel[panel["year"] == 2025].set_index("sido_sgg_nm")
p24 = panel[panel["year"] == 2024].set_index("sido_sgg_nm")
future = pd.DataFrame(index=p25.index)
future["prev_child_population"] = p25["child_population_0_12"]
future["prev_vehicle_count"] = p25["vehicle_count"]
future["prev_accident_count"] = p25["accident_count"]
future["prev_injury_count"] = p25["injury_count"]
future["prev_death_count"] = p25["death_count"]
future["prev_accident_per_10k_child"] = p25["accident_per_10k_child"]
future["prev_vehicles_per_100_child"] = p25["vehicles_per_100_child"]
future["accident_count_change_1y"] = p25["accident_count"] - p24["accident_count"]
future["accident_rate_change_1y"] = p25["accident_per_10k_child"] - p24["accident_per_10k_child"]
future["accident_count_mean_2y"] = (p25["accident_count"] + p24["accident_count"]) / 2
future["predicted_high_risk_probability_2026"] = best_model.predict_proba(future[feature_cols])[:, 1]
future = future.sort_values("predicted_high_risk_probability_2026", ascending=False).reset_index()
future.insert(0, "risk_rank_2026", range(1, len(future) + 1))
future.to_csv(BASE / "results" / "predicted_high_risk_2026_229.csv", index=False, encoding="utf-8-sig")
display(future[["risk_rank_2026", "sido_sgg_nm", "predicted_high_risk_probability_2026"]].head(20))


최종 모델: Logistic Regression


,risk_rank_2026,sido_sgg_nm,predicted_high_risk_probability_2026
0,1,부산광역시 중구,0.999923
1,2,강원특별자치도 양양군,0.996089
2,3,제주특별자치도 서귀포시,0.988549
3,4,경상북도 영덕군,0.985369
4,5,강원특별자치도 태백시,0.983254
5,6,전라남도 담양군,0.981943
6,7,충청북도 제천시,0.976913
7,8,충청남도 태안군,0.975390
8,9,전북특별자치도 무주군,0.966264
9,10,서울특별시 중구,0.955434


### 해석 주의
- 예측값은 실제 사고가 발생할 절대 확률이라고 단정하지 않는다.
- 모델은 시설 설치의 인과효과를 추정하지 않는다.
- 정책 활용 시 현장점검·예산 검토의 **우선순위 후보를 선별하는 보조도구**로 사용한다.
